<a href="https://colab.research.google.com/github/rm571222/dataholics-oracle-challenge/blob/main/notebooks/02_data_upload/nb7_upload_dados_complementares.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# NB7 — Carga no Oracle: Dados Complementares de Fontes Externas

**Projeto DATAHOLICS — FIAP Challenge | Parceria Oracle**

Este notebook documenta a carga de todas as tabelas de apoio que enriquecem o modelo com informações vindas de fontes externas ao SIH: o cadastro de estabelecimentos (CNES, em JSON), a descrição de diagnósticos (CID-10), a descrição dos domínios categóricos (sexo, raça/cor, tipo de AIH, caráter de internação, complexidade) e a classificação regional de saúde por município.

## Estrutura deste notebook
1. Bibliotecas e autenticação
2. Cadastro de estabelecimentos (CNES) — documento JSON
3. Tabelas de domínio (enums pequenos)
4. Tabela CID-10 (descrição de diagnósticos)
5. Tabela de Região de Saúde
6. Criação das foreign keys da tabela fato com os domínios

In [ ]:
!pip install -q oracledb requests pandas

from google.colab import files
uploaded = files.upload()  # selecione o Wallet_SPRINT02CHALLENGE.zip

import zipfile, os
wallet_path = '/content/wallet'
os.makedirs(wallet_path, exist_ok=True)
with zipfile.ZipFile(list(uploaded.keys())[0], 'r') as z:
    z.extractall(wallet_path)

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.5/2.5 MB 26.7 MB/s eta 0:00:00


Saving Wallet_SPRINT02CHALLENGE.zip to Wallet_SPRINT02CHALLENGE.zip


In [ ]:
import oracledb, pandas as pd, json, io, requests, zipfile as zf
import getpass

senha_banco = getpass.getpass("Senha do banco: ")
senha_wallet = getpass.getpass("Senha do wallet: ")

conn = oracledb.connect(
    user="ADMIN",
    password=senha_banco,
    dsn="sprint02challenge_high",
    config_dir=wallet_path,
    wallet_location=wallet_path,
    wallet_password=senha_wallet
)

cursor = conn.cursor()
cursor.execute("ALTER SESSION DISABLE PARALLEL DML")
cursor.execute("ALTER SESSION DISABLE PARALLEL QUERY")

def inserir_em_lotes(cursor, conn, sql, dados, tamanho_lote=5000):
    total_ignorados = 0
    erros_detalhados = []
    for i in range(0, len(dados), tamanho_lote):
        lote = dados[i:i + tamanho_lote]
        cursor.executemany(sql, lote, batcherrors=True)
        erros = cursor.getbatcherrors()
        for e in erros:
            erros_detalhados.append({'mensagem': str(e.message), 'linha': lote[e.offset]})
        total_ignorados += len(erros)
        conn.commit()
    if total_ignorados > 0:
        print(f'  ({total_ignorados} linhas ignoradas — ver erros_detalhados para causa exata)')
    return erros_detalhados

print("Conectado com sucesso!")

Senha do banco: ··········
Senha do wallet: ··········
Conectado com sucesso!


## 2. Cadastro de estabelecimentos (CNES) — documento JSON

Diferente das demais tabelas do modelo, o cadastro CNES é armazenado como coluna `JSON` nativa do Oracle — a peça que demonstra a convivência entre dado relacional e documento semiestruturado na arquitetura do projeto. Preserva o registro completo como vem da fonte, sem exigir definição de estrutura relacional fixa.

In [ ]:
cursor.execute("""
    BEGIN
        EXECUTE IMMEDIATE 'DROP TABLE T_SIH_ESTABELECIMENTO';
    EXCEPTION WHEN OTHERS THEN IF SQLCODE != -942 THEN RAISE; END IF;
    END;
""")
cursor.execute("""
    CREATE TABLE T_SIH_ESTABELECIMENTO (
        cd_hospital     VARCHAR2(10),
        json_cadastro   JSON,
        CONSTRAINT PK_SIH_ESTABELECIMENTO PRIMARY KEY (cd_hospital)
    )
""")
conn.commit()
print("Tabela T_SIH_ESTABELECIMENTO criada")

In [ ]:
resp = requests.get("https://s3.sa-east-1.amazonaws.com/ckan.saude.gov.br/CNES/cnes_estabelecimentos_json.zip")
z = zf.ZipFile(io.BytesIO(resp.content))
with z.open(z.namelist()[0]) as f:
    dados_cnes = json.load(f)

df_cnes = pd.DataFrame(dados_cnes)
df_cnes_sp = df_cnes[df_cnes['CO_UF'] == '35'].copy()
print(f"Estabelecimentos em SP: {len(df_cnes_sp)}")

INSERT_SQL_T3 = "INSERT INTO T_SIH_ESTABELECIMENTO (cd_hospital, json_cadastro) VALUES (:1, :2)"
registros = df_cnes_sp.to_dict(orient='records')
dados_t3 = [(str(r['CO_CNES']), json.dumps(r, ensure_ascii=False)) for r in registros]

inserir_em_lotes(cursor, conn, INSERT_SQL_T3, dados_t3)

cursor.execute("SELECT COUNT(*) FROM T_SIH_ESTABELECIMENTO")
print(f"\nTotal na T_SIH_ESTABELECIMENTO: {cursor.fetchone()[0]}")

## 3. Tabelas de domínio (enums pequenos)

Cinco tabelas dedicadas, uma por atributo categórico — decisão de modelagem que evita o anti-padrão de uma "tabela de domínio genérica" (chave composta por nome do domínio + código), que mistura semânticas diferentes e dificulta FKs específicas.

In [ ]:
ddls_dominio = {
    'T_SIH_TIPO_AIH': """CREATE TABLE T_SIH_TIPO_AIH (
        cd_tipo_aih NUMBER(1), ds_tipo_aih VARCHAR2(100),
        CONSTRAINT PK_SIH_TIPO_AIH PRIMARY KEY (cd_tipo_aih))""",
    'T_SIH_SEXO': """CREATE TABLE T_SIH_SEXO (
        sg_sexo VARCHAR2(2), ds_sexo VARCHAR2(50),
        CONSTRAINT PK_SIH_SEXO PRIMARY KEY (sg_sexo))""",
    'T_SIH_RACA_COR': """CREATE TABLE T_SIH_RACA_COR (
        cd_raca_cor VARCHAR2(2), ds_raca_cor VARCHAR2(50),
        CONSTRAINT PK_SIH_RACA_COR PRIMARY KEY (cd_raca_cor))""",
    'T_SIH_CARATER_INTERNACAO': """CREATE TABLE T_SIH_CARATER_INTERNACAO (
        cd_carater_internacao VARCHAR2(2), ds_carater_internacao VARCHAR2(150),
        CONSTRAINT PK_SIH_CARATER_INTERNACAO PRIMARY KEY (cd_carater_internacao))""",
    'T_SIH_COMPLEXIDADE': """CREATE TABLE T_SIH_COMPLEXIDADE (
        cd_complexidade VARCHAR2(2), ds_complexidade VARCHAR2(50),
        CONSTRAINT PK_SIH_COMPLEXIDADE PRIMARY KEY (cd_complexidade))""",
}

for nome, ddl in ddls_dominio.items():
    cursor.execute(f"""
        BEGIN
            EXECUTE IMMEDIATE 'DROP TABLE {nome}';
        EXCEPTION WHEN OTHERS THEN IF SQLCODE != -942 THEN RAISE; END IF;
        END;
    """)
    cursor.execute(ddl)
    conn.commit()
print("5 tabelas de domínio criadas")

As descrições vêm do dicionário de dados do CEM/USP (ver `fontes_dados_complementares.md`), com confirmação cruzada onde aplicável. O código `sg_sexo = '0'` é descrito como "Não informado" (não "Ignorado", conforme decisão de padronização de nomenclatura do projeto).

In [ ]:
dados_tipo_aih = [(1, 'AIH Normal'), (5, 'AIH de Longa Permanência / FPT')]
dados_sexo = [('0', 'Não informado'), ('1', 'Masculino'), ('3', 'Feminino')]
dados_raca_cor = [
    ('01', 'Branca'), ('02', 'Preta'), ('03', 'Parda'),
    ('04', 'Amarela'), ('05', 'Indígena'), ('99', 'Sem informação')
]
dados_carater_internacao = [
    ('01', 'Eletivo'), ('02', 'Urgência'),
    ('03', 'Acidente no local de trabalho ou a serviço da empresa'),
    ('04', 'Acidente no trajeto para o trabalho'),
    ('05', 'Outros tipos de acidente de trânsito'),
    ('06', 'Outros tipos de lesões e envenenamentos por agentes químicos ou físicos'),
]
dados_complexidade = [('01', 'Atenção Básica'), ('02', 'Média Complexidade'), ('03', 'Alta Complexidade')]

tabelas_dominio = [
    ('T_SIH_TIPO_AIH', dados_tipo_aih),
    ('T_SIH_SEXO', dados_sexo),
    ('T_SIH_RACA_COR', dados_raca_cor),
    ('T_SIH_CARATER_INTERNACAO', dados_carater_internacao),
    ('T_SIH_COMPLEXIDADE', dados_complexidade),
]

for nome_tabela, dados in tabelas_dominio:
    sql = f"INSERT INTO {nome_tabela} VALUES (:1, :2)"
    erros = inserir_em_lotes(cursor, conn, sql, dados)
    cursor.execute(f"SELECT COUNT(*) FROM {nome_tabela}")
    print(f"{nome_tabela}: {cursor.fetchone()[0]} registros, {len(erros)} erros")

## 4. Tabela CID-10 (descrição de diagnósticos)

Usamos a tabela de **subcategorias** (4 dígitos), que cobre tanto os códigos completos quanto, com um fallback (`código + '0'`), os códigos de categoria sem subdivisão.

In [ ]:
resp = requests.get("https://raw.githubusercontent.com/cleytonferrari/CidDataSus/master/CIDImport/Repositorio/Resources/CID-10-SUBCATEGORIAS.CSV")
df_cid_sub = pd.read_csv(io.BytesIO(resp.content), sep=';', encoding='latin1')
df_cid_sub = df_cid_sub[['SUBCAT', 'DESCRICAO']].rename(columns={'SUBCAT': 'cd_diagnostico', 'DESCRICAO': 'ds_diagnostico'})
df_cid_sub['cd_diagnostico'] = df_cid_sub['cd_diagnostico'].astype(str).str.strip()

print(f"Total de subcategorias CID-10: {len(df_cid_sub)}")

# O tamanho da coluna de descrição é definido dinamicamente pelo maior valor real da fonte
# (algumas descrições de causas externas de acidentes ultrapassam 250 caracteres)
maior_descricao = df_cid_sub['ds_diagnostico'].str.len().max()

cursor.execute("""
    BEGIN
        EXECUTE IMMEDIATE 'DROP TABLE T_SIH_CID10';
    EXCEPTION WHEN OTHERS THEN IF SQLCODE != -942 THEN RAISE; END IF;
    END;
""")
cursor.execute(f"""
    CREATE TABLE T_SIH_CID10 (
        cd_diagnostico   VARCHAR2(4),
        ds_diagnostico   VARCHAR2({maior_descricao + 20}),
        CONSTRAINT PK_SIH_CID10 PRIMARY KEY (cd_diagnostico)
    )
""")
conn.commit()

dados_cid = list(df_cid_sub.itertuples(index=False, name=None))
inserir_em_lotes(cursor, conn, "INSERT INTO T_SIH_CID10 (cd_diagnostico, ds_diagnostico) VALUES (:1, :2)", dados_cid)

cursor.execute("SELECT COUNT(*) FROM T_SIH_CID10")
print(f"Total na T_SIH_CID10: {cursor.fetchone()[0]}")

**Nota:** não é criada FK entre `T_SIH_INTERNACAO.cd_diagnostico_principal` e esta tabela — a cobertura de 99,99% (não 100%) tornaria uma FK rígida um risco de rejeitar registros legítimos (códigos criados após 2008, versão desta tabela). A relação permanece lógica, resolvida via `JOIN` na hora da consulta.

## 5. Tabela de Região de Saúde

Resolve a lacuna de "visão regional" identificada na exploração (NB1) — o SIH só tem código de município, não região. Fonte: Ministério da Saúde, Regionalização no SUS (download manual, portal bloqueia acesso automatizado).

In [ ]:
from google.colab import files
uploaded = files.upload()  # selecione o CSV de região de saúde (Ministério da Saúde)

nome_arquivo = list(uploaded.keys())[0]
df_regiao_raw = pd.read_csv(nome_arquivo)

df_regiao = df_regiao_raw[df_regiao_raw['UF'] == 'Sao Paulo'].copy()
df_regiao = df_regiao.rename(columns={
    'Codigo Municipio': 'cd_municipio', 'Municipio': 'nm_municipio',
    'Codigo Regiao de Saude': 'cd_regiao_saude', 'Regiao de Saude': 'nm_regiao_saude',
    'Codigo Macrorregiao de Saude': 'cd_macrorregiao', 'Macrorregiao de Saude': 'nm_macrorregiao'
})
df_regiao['cd_municipio'] = df_regiao['cd_municipio'].astype(str)
df_regiao['cd_regiao_saude'] = df_regiao['cd_regiao_saude'].astype(str)
df_regiao['cd_macrorregiao'] = df_regiao['cd_macrorregiao'].astype(str)

print(f"Municípios de SP: {len(df_regiao)}")

In [ ]:
cursor.execute("""
    BEGIN
        EXECUTE IMMEDIATE 'DROP TABLE T_SIH_REGIAO_SAUDE';
    EXCEPTION WHEN OTHERS THEN IF SQLCODE != -942 THEN RAISE; END IF;
    END;
""")
cursor.execute("""
    CREATE TABLE T_SIH_REGIAO_SAUDE (
        cd_municipio      VARCHAR2(6),
        nm_municipio      VARCHAR2(100),
        cd_regiao_saude   VARCHAR2(10),
        nm_regiao_saude   VARCHAR2(100),
        cd_macrorregiao   VARCHAR2(10),
        nm_macrorregiao   VARCHAR2(100),
        CONSTRAINT PK_SIH_REGIAO_SAUDE PRIMARY KEY (cd_municipio)
    )
""")
conn.commit()

COLUNAS_REGIAO = ['cd_municipio', 'nm_municipio', 'cd_regiao_saude', 'nm_regiao_saude', 'cd_macrorregiao', 'nm_macrorregiao']
INSERT_SQL_REGIAO = f"INSERT INTO T_SIH_REGIAO_SAUDE ({','.join(COLUNAS_REGIAO)}) VALUES ({','.join([':'+str(i+1) for i in range(len(COLUNAS_REGIAO))])})"

dados_regiao = list(df_regiao[COLUNAS_REGIAO].itertuples(index=False, name=None))
inserir_em_lotes(cursor, conn, INSERT_SQL_REGIAO, dados_regiao)

cursor.execute("SELECT COUNT(*) FROM T_SIH_REGIAO_SAUDE")
print(f"Total na T_SIH_REGIAO_SAUDE: {cursor.fetchone()[0]}")

## 6. Criação das foreign keys da tabela fato com os domínios

Antes de criar cada FK, confirmamos que não há valor na tabela fato sem correspondência no domínio — evita erro de "chave mãe não localizada".

In [ ]:
checagens = [
    ("cd_tipo_aih", "T_SIH_TIPO_AIH", "cd_tipo_aih"),
    ("sg_sexo", "T_SIH_SEXO", "sg_sexo"),
    ("cd_raca_cor", "T_SIH_RACA_COR", "cd_raca_cor"),
    ("cd_carater_internacao", "T_SIH_CARATER_INTERNACAO", "cd_carater_internacao"),
    ("cd_complexidade", "T_SIH_COMPLEXIDADE", "cd_complexidade"),
]
for col_fato, tabela_dom, col_dom in checagens:
    cursor.execute(f"""
        SELECT COUNT(DISTINCT i.{col_fato}) FROM T_SIH_INTERNACAO i
        LEFT JOIN {tabela_dom} d ON i.{col_fato} = d.{col_dom}
        WHERE d.{col_dom} IS NULL
    """)
    print(f"{col_fato}: {cursor.fetchone()[0]} valores sem correspondência em {tabela_dom}")

In [ ]:
fks = [
    ("FK_SIH_INT_TIPO_AIH", "cd_tipo_aih", "T_SIH_TIPO_AIH", "cd_tipo_aih"),
    ("FK_SIH_INT_SEXO", "sg_sexo", "T_SIH_SEXO", "sg_sexo"),
    ("FK_SIH_INT_RACA_COR", "cd_raca_cor", "T_SIH_RACA_COR", "cd_raca_cor"),
    ("FK_SIH_INT_CARATER", "cd_carater_internacao", "T_SIH_CARATER_INTERNACAO", "cd_carater_internacao"),
    ("FK_SIH_INT_COMPLEXIDADE", "cd_complexidade", "T_SIH_COMPLEXIDADE", "cd_complexidade"),
]
for nome_fk, col_fato, tabela_dom, col_dom in fks:
    try:
        cursor.execute(f"""
            ALTER TABLE T_SIH_INTERNACAO ADD CONSTRAINT {nome_fk}
            FOREIGN KEY ({col_fato}) REFERENCES {tabela_dom}({col_dom})
        """)
        conn.commit()
        print(f"{nome_fk} criada")
    except Exception as e:
        print(f"{nome_fk} falhou: {e}")

**Nota técnica:** a FK de `cd_tipo_aih` exigiu ajuste de tipo — a tabela de domínio foi criada com `NUMBER(1)` para bater com o tipo da coluna na tabela fato (ambos numéricos, evitando o erro `ORA-02267` de incompatibilidade de tipo entre colunas referenciadas).

## Conclusão — NB7

Sete tabelas de apoio foram carregadas: cadastro CNES (153.366 estabelecimentos, formato JSON), 5 tabelas de domínio (100% de cobertura contra a tabela fato, com FKs ativas), CID-10 (12.451 códigos, relação lógica sem FK física dada a cobertura de 99,99%) e Região de Saúde (645 municípios). Essas tabelas fecham o modelo de enriquecimento do projeto, cobrindo todas as colunas de código identificadas na fase de exploração.